# Running All End-to-End Benchmarks 

In [ ]:
import math
from common import *
from experiment import *
from data.http import HTTPExperiment
from treatments.picoquic import treatment_map
from treatments.network_settings import *

## HTTP File Transfer

### Network Settings

In [ ]:
N_TRIALS = 1

labels_cubic = [
    f'picoquic_iblt_{ACK_DELAY_WIFI}ms_hint',
    f'picoquic_iblt_{ACK_DELAY_SAT}ms_hint',
    f'picoquic_iblt_{ACK_DELAY_CELL}ms_hint',
    'picoquic',
    'picoquic_split',
    'picoquic_rtunnel_retx7',
    'picoquic_rtunnel_retx7_ordered32',
]

labels_bbr = [l + "_bbr" for l in labels_cubic]

# Protocol and proxy configurations
treatments_cubic = [treatment_map(label) for label in labels_cubic]
treatments_bbr = [treatment_map(label) for label in labels_bbr]

# Network settings
LOSS1_VALUES = [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16, 18]
networks = network_settings(LOSS1_VALUES)

max_networks = {'picoquic' : 10, 'picoquic_rtunnel_retx' : 14}

# Data transfer size; aim for 10s transfer under optimal conditions
BPS = 125000 # 1Mbps = 125,000 bytes
def DATA_SIZE_(bottleneck_bw=20, time_s=10):
    return bottleneck_bw * BPS * time_s

def DATA_SIZE(network: NetworkSetting, time_s=10):
    return DATA_SIZE_(bottleneck_bw=min(network.get('bw1'), network.get('bw2')), time_s=time_s)

### Plotting and Data Collection Helpers

In [21]:
def plot_loss_vs_metric_line(data, title, ylabel, ncol=3, ylim=0, delta=25, pdf=None, style=False, ge=False):
    plt.figure(figsize=(6, 3))
    
    keys = data.treatments
    data_size = data.data_sizes[0]

    # Plot each label
    labels = []
    for key in keys:
        xs = []
        ys_raw = []

        for network in data.network_settings:
            if data_size not in data.data[key][network]:
                continue
            network_setting = data.exp.get_network_setting(network)
            
            if ge:
                xs.append(ge_to_burst_size(network_setting.settings['ge']))
            else:
                xs.append(float(network_setting.settings['loss1']))
                
            ys_raw.append(data.data[key][network][data_size])

        ys = [y.p(50) for y in ys_raw]
        yerr_lower = [y.p(50) - y.p(50-delta) for y in ys_raw]
        yerr_upper = [y.p(50+delta) - y.p(50) for y in ys_raw]
        if style:
            label = LABEL_MAP[key]
            sty = STYLE[label]
            title = None
            ylim = None
            plt.errorbar(xs, ys, yerr=(yerr_lower, yerr_upper), marker=sty.marker, markersize=sty.markersize, capsize=5, label=label, color=sty.color, linestyle=sty.linestyle)
        else:
            label = key
            plt.errorbar(xs, ys, yerr=(yerr_lower, yerr_upper), marker='.', capsize=5, label=label)
        print(label, list(zip(xs, ys)))
        labels.append(label)

    plt.title(title)
    if ge:
        plt.xlabel('Expected Length (Packets) of Lossy Burst')
    else: 
        plt.xlabel('Loss % Near Data Receiver')
    plt.ylabel(ylabel)
    plt.grid()
    plt.xlim(0)
    plt.ylim(ylim)
    plot_title_and_legend(title, labels, base_height=1.2, row_height=0.07, title_height=0.08, ncol=ncol)
    if pdf:
        save_pdf(pdf)
    plt.show()

def collect_loss_vs_metric_data(treatments, network_settings, data_size, n=10, max_networks=max_networks, execute=False, min_i=1):
    if execute:
        num_trials_range = range(min_i, n+1)
    else:
        num_trials_range = [n]
    for i in num_trials_range:
        exp = HTTPExperiment(num_trials=i, treatments=treatments, network_settings=network_settings, data_sizes=[data_size])
        raw_data = exp.to_raw_data(execute=execute, max_networks=max_networks)
    return raw_data

def plot_loss_vs_throughput(raw_data, title=None, ncol=2, pdf=None, style=False, ge=False):
    plottable_data = raw_data.to_plottable_data('throughput_mbps')
    if not title:
        title = f'{data_size_str(plottable_data.data_sizes[0])} ({plottable_data.network_settings[0]})'
    ylabel = 'Goodput (Mbit/s)'
    plot_loss_vs_metric_line(plottable_data, title, ncol=ncol, ylabel=ylabel, pdf=pdf, style=style, ge=ge)
    

### Results

In [22]:
def exec_single(treatments, network_settings, label, execute=True, ge=False):
    loss_vs_throughput_data = collect_loss_vs_metric_data(
        n=N_TRIALS,
        execute=execute,
        treatments=treatments,
        max_networks={},
        network_settings=network_settings,
        data_size=DATA_SIZE(network_settings[0]),
    )
    plot_loss_vs_throughput(loss_vs_throughput_data, ncol=3, style=True, pdf=f'../figures/{label}.pdf', ge=ge)

CUBIC

In [ ]:
# CUBIC, WIFI network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_cubic if ('iblt' not in t.label() or f'{ACK_DELAY_WIFI}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['wifi'], label="http_benchmark_wifi_cubic", execute=True)

6
sudo -E python3 emulation/main.py --bw1 50 --bw2 20 --delay1 2 --delay2 30 --loss1 8 --loss2 0 -t 1 --label picoquic --network-statistics picoquic --client-quacker -n 25000000 91.41893196105957
sudo -E python3 emulation/main.py --bw1 50 --bw2 20 --delay1 2 --delay2 30 --loss1 10 --loss2 0 -t 1 --label picoquic --network-statistics picoquic --client-quacker -n 25000000 125.11037278175354
sudo -E python3 emulation/main.py --bw1 50 --bw2 20 --delay1 2 --delay2 30 --loss1 12 --loss2 0 -t 1 --label picoquic --network-statistics picoquic --client-quacker -n 25000000 

In [ ]:
# CUBIC, satellite network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_cubic if ('iblt' not in t.label() or f'{ACK_DELAY_SAT}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['sat'], label="http_benchmark_sat_cubic", execute=True)

In [ ]:
# CUBIC, cellular network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_cubic if ('iblt' not in t.label() or f'{ACK_DELAY_CELL}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['cell'], label="http_benchmark_cell_cubic", execute=True)

BBRv3

In [ ]:
# BBR, WIFI network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_bbr if ('iblt' not in t.label() or f'{ACK_DELAY_WIFI}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['wifi'], label="http_benchmark_wifi_bbr3", execute=True)

In [ ]:
# BBR, satellite network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_bbr if ('iblt' not in t.label() or f'{ACK_DELAY_SAT}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['sat'], label="http_benchmark_sat_bbr3", execute=True)

In [ ]:
# BBR, cellular network
# Exclude treatments for different ACK delays
treatments = [t for t in treatments_bbr if ('iblt' not in t.label() or f'{ACK_DELAY_CELL}ms' in t.label())]
exec_single(treatments=treatments, network_settings=networks['sat'], label="http_benchmark_cell_bbr3", execute=True)